# 17. Global threshold для иерархического Span NER

Порог выбирается **только на validation** по максимальному strict entity-level micro-F1 исходных классов `ACT`, `BIN`, `CMP`, `QUA`, `ECO`, `SOC`, `MET`, `INST`. Используется fine-head иерархической модели. Test split в этом ноутбуке не загружается.

Encoder выполняет один inference-проход; все значения threshold проверяются по уже рассчитанным вероятностям. Gold-метки в модель не передаются: fine/coarse/contrastive losses во время калибровки не вычисляются. На Google Drive записываются только небольшие `threshold_calibration.csv` и `threshold_calibration.json`.

> Если в текущей Colab-сессии уже возникал `device-side assert triggered`, сначала выполните **Runtime → Restart session**. Повреждённый CUDA-контекст нельзя восстановить повторным запуском ячейки.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import runpy

PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
EXPERIMENT_CONFIG = PROJECT_DIR / 'configs/experiments/hierarchical_span_ner_corrected_v1.yaml'
OUTPUT_DIR = PROJECT_DIR / 'results/hierarchical_span_ner_corrected_v1/seed_42'
CHECKPOINT = OUTPUT_DIR / 'checkpoints/best'
BOOTSTRAP = PROJECT_DIR / 'colab_bootstrap.py'

for required_path in (EXPERIMENT_CONFIG, CHECKPOINT / 'config.json', CHECKPOINT / 'model.safetensors', BOOTSTRAP):
    if not required_path.exists():
        raise FileNotFoundError(
            f'Не найден {required_path}. Обновите проект и сначала выполните 16_train_hierarchical_span_ner.ipynb.'
        )

bootstrap_project = runpy.run_path(str(BOOTSTRAP))['bootstrap_project']
bootstrap_project(PROJECT_DIR)

In [ ]:
from inspect import signature
from rurebus_ie.training.span_trainer import SpanNerTrainer

parameters = signature(SpanNerTrainer._collect_candidates).parameters
if 'include_labels' not in parameters:
    raise RuntimeError(
        'В Google Drive находится устаревший span_trainer.py. '
        'Скопируйте текущий проект, перезапустите runtime и начните notebook заново.'
    )
print('Calibration inference без labels поддерживается.')

In [ ]:
import json

with (CHECKPOINT / 'config.json').open(encoding='utf-8') as stream:
    checkpoint_config = json.load(stream)

model_type = checkpoint_config.get('model_type')
training_stage = checkpoint_config.get('training_stage')
print('model_type:', model_type)
print('training_stage:', training_stage)

if model_type != 'rurebus_hierarchical_span_ner':
    raise ValueError('CHECKPOINT не является иерархической Span NER моделью.')
if training_stage != 'fine':
    raise ValueError('Для калибровки требуется лучший checkpoint fine-этапа.')

In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    try:
        probe = torch.zeros(1, device='cuda')
        del probe
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    except RuntimeError as error:
        raise RuntimeError(
            'CUDA-контекст повреждён предыдущей device-side assert. '
            'Выполните Runtime → Restart session и начните notebook заново.'
        ) from error
else:
    print('Предупреждение: калибровка выполнится на CPU, но будет заметно медленнее.')

In [ ]:
from rurebus_ie.training import calibrate_hierarchical_span_threshold_experiment

# Текущий threshold=0.50 даёт избыточный recall. Проверяем достаточно
# широкий диапазон; test при выборе порога не используется.
THRESHOLDS = [round(0.30 + step * 0.01, 2) for step in range(69)]  # 0.30 ... 0.98

calibration = calibrate_hierarchical_span_threshold_experiment(
    EXPERIMENT_CONFIG,
    thresholds=THRESHOLDS,
    project_root=PROJECT_DIR,
    checkpoint_dir=CHECKPOINT,
    output_dir_override=OUTPUT_DIR,
)

print(f'Оптимальный global threshold: {calibration.best_threshold:.2f}')
print(f'Validation precision: {calibration.best_metrics.precision:.4f}')
print(f'Validation recall: {calibration.best_metrics.recall:.4f}')
print(f'Validation micro-F1: {calibration.best_metrics.micro_f1:.4f}')
print(f'Validation macro-F1: {calibration.best_metrics.macro_f1:.4f}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

threshold_table = pd.DataFrame(calibration.rows)
best_index = threshold_table['micro_f1'].idxmax()
display(threshold_table.iloc[max(0, best_index - 5):best_index + 6])

ax = threshold_table.plot(
    x='threshold',
    y=['precision', 'recall', 'micro_f1', 'macro_f1'],
    figsize=(11, 6),
    grid=True,
)
ax.axvline(calibration.best_threshold, color='black', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)
ax.set_title('Hierarchical Span NER: global threshold на validation')
plt.show()

In [ ]:
per_class = pd.DataFrame(calibration.best_metrics.per_class).T.sort_values('f1')
display(per_class)

best_row = threshold_table.loc[best_index]
print('Предсказанных сущностей:', int(best_row['predicted_entities']))
print('JSON-отчёт:', OUTPUT_DIR / 'threshold_calibration.json')
print('CSV-таблица:', OUTPUT_DIR / 'threshold_calibration.csv')
print('Следующий этап: применить сохранённый threshold при однократной оценке test.')